# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/ismgonza/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/ismgonza/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
# This code populates the knowledge graph with our documents

from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'fab372'. Skipping!
Property 'summary' already exists in node '7e5ef9'. Skipping!
Property 'summary' already exists in node 'd48f45'. Skipping!
Property 'summary' already exists in node 'bcd047'. Skipping!
Property 'summary' already exists in node '4d21a3'. Skipping!
Property 'summary' already exists in node '69a936'. Skipping!
Property 'summary' already exists in node '744718'. Skipping!
Property 'summary' already exists in node 'ec4991'. Skipping!
Property 'summary' already exists in node 'bee4cb'. Skipping!
Property 'summary' already exists in node '047203'. Skipping!
Property 'summary' already exists in node '385b16'. Skipping!
Property 'summary' already exists in node '4aeb82'. Skipping!
Property 'summary' already exists in node '60e4fd'. Skipping!
Property 'summary' already exists in node '1deef4'. Skipping!
Property 'summary' already exists in node '86e91e'. Skipping!
Property 'summary' already exists in node '6e61d4'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '4d21a3'. Skipping!
Property 'summary_embedding' already exists in node 'fab372'. Skipping!
Property 'summary_embedding' already exists in node 'bcd047'. Skipping!
Property 'summary_embedding' already exists in node 'd48f45'. Skipping!
Property 'summary_embedding' already exists in node '69a936'. Skipping!
Property 'summary_embedding' already exists in node '7e5ef9'. Skipping!
Property 'summary_embedding' already exists in node 'bee4cb'. Skipping!
Property 'summary_embedding' already exists in node 'ec4991'. Skipping!
Property 'summary_embedding' already exists in node '385b16'. Skipping!
Property 'summary_embedding' already exists in node '047203'. Skipping!
Property 'summary_embedding' already exists in node '744718'. Skipping!
Property 'summary_embedding' already exists in node '4aeb82'. Skipping!
Property 'summary_embedding' already exists in node '60e4fd'. Skipping!
Property 'summary_embedding' already exists in node '86e91e'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 712)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 712)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

-----------------------------------------------------------------------------------------------------------------------------------------------------------------
##### ANSWER

1. SingleHop (50%): Makes simple, direct questions that need just ONE piece of info from ONE document
2. MultiHopAbstract (25%): Makes complex questions that need info from MULTIPLE documents and require summarizing or generalizing
3. MultiHopSpecific (25%): Makes complex questions that need info from MULTIPLE documents to find a specific fact or detail

Simple way to remember:

SingleHop = easy, one source
MultiHop Abstract = hard, multiple sources, big picture answer
MultiHop Specific = hard, multiple sources, specific answer

-----------------------------------------------------------------------------------------------------------------------------------------------------------------

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the significance of November 2022 in t...,[Introduction ChatGPT launched in November 202...,Introduction ChatGPT launched in November 2022.,single_hop_specifc_query_synthesizer
1,How does Claude relate to the usage patterns o...,[Table 1: ChatGPT daily message counts (millio...,Claude is mentioned in the context of comparin...,single_hop_specifc_query_synthesizer
2,As a Career Analyst examining occupational dat...,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 presents var...,single_hop_specifc_query_synthesizer
3,Personal Reflection how does ChatGPT impact wo...,[Conclusion This paper studies the rapid growt...,The context states that about 70% of ChatGPT c...,single_hop_specifc_query_synthesizer
4,Based on the data showing that non-work messag...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data indicates that non-work message volum...,multi_hop_abstract_query_synthesizer
5,how message classification and taxonomy show A...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,the context explains that most ChatGPT message...,multi_hop_abstract_query_synthesizer
6,How does the global economic impact of ChatGPT...,[<1-hop>\n\nConclusion This paper studies the ...,The global economic impact of ChatGPT is signi...,multi_hop_abstract_query_synthesizer
7,"Based on the information that by July 2025, 70...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The widespread adoption of ChatGPT, with over ...",multi_hop_specific_query_synthesizer
8,Based on the data about ChatGPT's usage in the...,[<1-hop>\n\nConclusion This paper studies the ...,"The data indicates that by July 2025, more tha...",multi_hop_specific_query_synthesizer
9,"Whos Handa et al., 2025 and how does it relate...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,"Handa et al., 2025 is a referenced study that ...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
#for this one we just need:
# from langchain_community.document_loaders import DirectoryLoader
# from langchain_community.document_loaders import PyMuPDFLoader


# path = "data/"
# loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
# docs = loader.load()

# from ragas.llms import LangchainLLMWrapper
# from ragas.embeddings import LangchainEmbeddingsWrapper
# from langchain_openai import ChatOpenAI
# from langchain_openai import OpenAIEmbeddings
# generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
# generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '1ae546'. Skipping!
Property 'summary' already exists in node 'ba2027'. Skipping!
Property 'summary' already exists in node '0d77fc'. Skipping!
Property 'summary' already exists in node 'c8ee57'. Skipping!
Property 'summary' already exists in node '75a894'. Skipping!
Property 'summary' already exists in node '364e17'. Skipping!
Property 'summary' already exists in node '0579a9'. Skipping!
Property 'summary' already exists in node '5019ab'. Skipping!
Property 'summary' already exists in node 'cc0319'. Skipping!
Property 'summary' already exists in node '7298a3'. Skipping!
Property 'summary' already exists in node '82189c'. Skipping!
Property 'summary' already exists in node '140301'. Skipping!
Property 'summary' already exists in node '49f155'. Skipping!
Property 'summary' already exists in node '117d29'. Skipping!
Property 'summary' already exists in node 'c3893b'. Skipping!
Property 'summary' already exists in node '4512f4'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '364e17'. Skipping!
Property 'summary_embedding' already exists in node '1ae546'. Skipping!
Property 'summary_embedding' already exists in node '75a894'. Skipping!
Property 'summary_embedding' already exists in node '0579a9'. Skipping!
Property 'summary_embedding' already exists in node 'c8ee57'. Skipping!
Property 'summary_embedding' already exists in node 'ba2027'. Skipping!
Property 'summary_embedding' already exists in node 'cc0319'. Skipping!
Property 'summary_embedding' already exists in node '7298a3'. Skipping!
Property 'summary_embedding' already exists in node '0d77fc'. Skipping!
Property 'summary_embedding' already exists in node '5019ab'. Skipping!
Property 'summary_embedding' already exists in node '82189c'. Skipping!
Property 'summary_embedding' already exists in node '140301'. Skipping!
Property 'summary_embedding' already exists in node 'c3893b'. Skipping!
Property 'summary_embedding' already exists in node '117d29'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,"What does Bick et al., 2024 say about ChatGPT'...",[Introduction ChatGPT launched in November 202...,"Bick et al., 2024 reports that by July 2025, 1...",single_hop_specifc_query_synthesizer
1,how much ChatGPT messages are work related and...,[Table 1: ChatGPT daily message counts (millio...,"According to the context, ChatGPT messages are...",single_hop_specifc_query_synthesizer
2,What is Appendix D about in relation to ChatGP...,[Variation by Occupation Figure 23 presents va...,Appendix D presents variation in ChatGPT usage...,single_hop_specifc_query_synthesizer
3,How does the fact that ChatGPT has reached ove...,[Conclusion This paper studies the rapid growt...,"By July 2025, ChatGPT had been used weekly by ...",single_hop_specifc_query_synthesizer
4,How do the variation in ChatGPT usage across d...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The variation in ChatGPT usage across occupati...,multi_hop_abstract_query_synthesizer
5,How does the rapid global diffusion of ChatGPT...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT's rapid adoption—reachin...",multi_hop_abstract_query_synthesizer
6,How do work and non-work usag differ in ChatGP...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The context indicates that about 70% of ChatGP...,multi_hop_abstract_query_synthesizer
7,How does the impact of occupation on ChatGPT i...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context shows that users in highly paid pr...,multi_hop_abstract_query_synthesizer
8,How does the rapid growth of ChatGPT since its...,[<1-hop>\n\nConclusion This paper studies the ...,"Since its launch in November 2022, ChatGPT exp...",multi_hop_specific_query_synthesizer
9,Whi is the date in July 2025 that ChatGPT had ...,[<1-hop>\n\nConclusion This paper studies the ...,"In July 2025, ChatGPT had 700 million users, w...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

# The flow step-by-step:

# Takes the question
# Sends question to retriever → gets 10 relevant chunks (context)
# Fills the prompt template with context + question
# Sends to LLM → gets answer
# Extracts just the text from response

In [27]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways both at work and outside of work. Specifically, AI is being used to perform workplace tasks by augmenting or automating human labor. Users engage with AI in different modes of interaction categorized as Asking (seeking information or advice), Doing (producing outputs such as writing, software code, spreadsheets, and other digital products), and Expressing (self-expression, including relationships, personal reflection, games, and role play). \n\nGenerative AI is noted for its flexibility and ability to produce diverse digital products, distinguishing it from traditional web search engines. There are also implications for occupational productivity, where AI serves either as a co-worker producing output or as a co-pilot providing advice and enhancing human problem-solving.\n\nIn summary, people are using AI to:\n\n- Augment or automate work tasks\n- Generate writing, software

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [30]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"

            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

# If by any reason i dont want to define custom criteria, i just use the built-in evaluators without custom definitions, for example

# qa_evaluator = LangChainStringEvaluator("qa", config={"llm": eval_llm})
# context_evaluator = LangChainStringEvaluator("context_qa", config={"llm": eval_llm})

# Common default evaluators:

# "qa" - correctness (you're already using this one!)
# "cot_qa" - correctness with chain-of-thought reasoning
# "context_qa" - checks if answer uses provided context
# "labeled_score_string" - scores on a scale

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: Checks basic correctness - does the answer match the expected answer?
- `labeled_helpfulness_evaluator`: Is the answer actually helpful, comparing it to the reference answer?. Makes sure it is not just correct, but usefully correct
- `dopeness_evaluator`: Is the answer engaging/interesting or boring and generic? in orther words checks the "vibe" of the answer

## LangSmith Evaluation

In [31]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'crushing-part-32' at:
https://smith.langchain.com/o/bb67da1a-f981-488f-9c09-ce7a028c911d/datasets/cd6ca13d-afd1-4e2a-a777-4062688fa96d/compare?selectedSessions=5537da3f-7c64-411e-a641-538327a63654




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,whats hapening in July 2025 with ChatGPT usage...,"By July 2025, ChatGPT had more than 700 millio...",None,"In July 2025, ChatGPT had been used weekly by ...",1,1,0,4.699659,b96f3eee-4324-45e3-b5ed-da89c963a7db,ddf0195e-32a7-4906-9152-5c20c1b1bd3d
1,"Considering the rapid growth of ChatGPT, which...","Based on the provided context, by July 2025, C...",None,"By July 2025, ChatGPT users collectively sent ...",1,1,0,10.388054,d1b84333-bdfa-4a3d-846a-486b380878e3,fecebbb1-31d3-4c97-b09c-f906c5237d86
2,Whi is the date in July 2025 that ChatGPT had ...,The date in July 2025 when ChatGPT had more th...,None,"In July 2025, ChatGPT had 700 million users, w...",1,1,0,4.152442,9d9b217f-63dd-42ff-a799-fcba21f647a0,90cbfa97-ec42-40f6-b9c9-6c8bf95c6655
3,How does the rapid growth of ChatGPT since its...,The rapid growth of ChatGPT since its launch i...,None,"Since its launch in November 2022, ChatGPT exp...",1,1,0,11.056172,c36f79e0-2745-4f5e-87aa-1911dc46ee00,10c52003-30b3-4127-bf1c-4aa77045ef08
4,How does the impact of occupation on ChatGPT i...,"Based on the provided context, occupation sign...",None,The context shows that users in highly paid pr...,1,1,0,4.635352,e8dec195-0372-42d4-894c-3c15f1593395,345c867d-7092-43ef-8e0c-12a9ca1776c9
5,How do work and non-work usag differ in ChatGP...,Work and non-work usage of ChatGPT differ in t...,None,The context indicates that about 70% of ChatGP...,1,1,0,4.275890,c16bfaf9-3041-4087-9cfa-21d7137b6ffc,3cd330a6-e797-4fc9-b34d-b16de6b0acbc
6,How does the rapid global diffusion of ChatGPT...,"The rapid global diffusion of ChatGPT, reachin...",None,"By July 2025, ChatGPT's rapid adoption—reachin...",1,1,0,5.137675,fa51f599-9af3-4cb5-ab29-7c9053407858,8c0af3e5-2d94-4e79-bef9-6b822f514a23
7,How do the variation in ChatGPT usage across d...,The variation in ChatGPT usage across differen...,None,The variation in ChatGPT usage across occupati...,1,1,0,5.322477,ea8552d8-401a-4880-a037-7004adc8398f,52da0999-6d3f-4212-8f88-81c112787c76
8,How does the fact that ChatGPT has reached ove...,"By July 2025, ChatGPT had over 700 million wee...",None,"By July 2025, ChatGPT had been used weekly by ...",1,1,0,5.485821,19ddd676-6939-476b-8516-bd1dfcdf948e,b07341c9-4f3a-4495-8532-8c294e7fec66
9,What is Appendix D about in relation to ChatGP...,Appendix D contains a full report of GWA (Gene...,None,Appendix D presents variation in ChatGPT usage...,1,0,0,1.532759,ecbee565-4bd7-4f1f-9c07-3134263e7a11,f7ef9de7-0cc3-4751-b5ce-593d5b30eadc


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [32]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [33]:
rag_documents = docs

In [34]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

-------------------------------------------------------------------------------------------------------------------
#### ANSWER
To experiment whether larger context chunks help the RAG give better answers

-------------------------------------------------------------------------------------------------------------------

In [35]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

-------------------------------------------------------------------------------------------------------------------
#### ANSWER

Larger model = better quality embeddings = potentially better retrieval accuracy

-------------------------------------------------------------------------------------------------------------------

In [36]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [37]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [38]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [39]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Yo, let’s crank this up to eleven! Based on the killer context you dropped, people ain’t just punching buttons; they’re leveraging AI like ChatGPT as a secret weapon in their professional hustle. Here’s the scoop:\n\nThey’re not just handing off tasks to AI—they’re using it as their slick advisor and brainy research assistant, a digital sidekick leveling up decision-making in knowledge-heavy gigs. This isn’t just “automation,” it’s augmentation, baby. When decision quality rockets, worker output blasts off—meaning smarter moves, faster wins, and more $$$ stacking in the real world.\n\nPlus, the economic vibes are astronomical: U.S. users would demand a cool $98 each to skip AI for a whole month, pointing to a mind-blowing $97 billion surplus a year. That’s wild value creation, straight from AI-powered productivity and decision wizardry.\n\nSo in a nutshell? People are cashing in by using AI to boost their brainpower, crush decision paralysis, and turbocharge their know-how to make ban

Finally, we can evaluate the new chain on the same test set!

In [40]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'warm-collar-60' at:
https://smith.langchain.com/o/bb67da1a-f981-488f-9c09-ce7a028c911d/datasets/cd6ca13d-afd1-4e2a-a777-4062688fa96d/compare?selectedSessions=366d3a91-6703-458b-89c9-63efb56e6745




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,whats hapening in July 2025 with ChatGPT usage...,"Alright, buckle up—here’s the juicy lowdown on...",None,"In July 2025, ChatGPT had been used weekly by ...",1,1,1,8.094437,b96f3eee-4324-45e3-b5ed-da89c963a7db,dbbacd51-9a2d-4579-bfd6-3fe7f19c2405
1,"Considering the rapid growth of ChatGPT, which...","Yo, strap in — by July 2025, ChatGPT wasn’t ju...",None,"By July 2025, ChatGPT users collectively sent ...",1,1,1,7.907378,d1b84333-bdfa-4a3d-846a-486b380878e3,aa582aaf-f121-4282-9a42-26827514c1be
2,Whi is the date in July 2025 that ChatGPT had ...,"Alright, strap in for the turbocharged lowdown...",None,"In July 2025, ChatGPT had 700 million users, w...",1,1,1,5.677150,9d9b217f-63dd-42ff-a799-fcba21f647a0,64ee5403-cbb5-457c-b933-6eaa93c31de5
3,How does the rapid growth of ChatGPT since its...,"Alright, strap in for this wild ride through t...",None,"Since its launch in November 2022, ChatGPT exp...",1,1,1,8.669506,c36f79e0-2745-4f5e-87aa-1911dc46ee00,b9eab076-4ca5-4273-a443-19416d33d2bb
4,How does the impact of occupation on ChatGPT i...,"Yo, buckle up—here’s the 411 straight from the...",None,The context shows that users in highly paid pr...,1,1,1,7.082511,e8dec195-0372-42d4-894c-3c15f1593395,7afe3465-8503-40d9-baeb-ad0d8af1d710
5,How do work and non-work usag differ in ChatGP...,"Yo, here’s the lowdown dripping in pure dopene...",None,The context indicates that about 70% of ChatGP...,1,1,1,8.191932,c16bfaf9-3041-4087-9cfa-21d7137b6ffc,e2020961-bc3a-428d-b25b-ff1abdcf10ef
6,How does the rapid global diffusion of ChatGPT...,"Yo, here’s the deal—ChatGPT’s turbocharged tak...",None,"By July 2025, ChatGPT's rapid adoption—reachin...",1,1,1,7.270588,fa51f599-9af3-4cb5-ab29-7c9053407858,ab0857c9-3d36-4918-ae9a-60c4bfc4c9b7
7,How do the variation in ChatGPT usage across d...,"Alright, here’s the lowdown on how ChatGPT vib...",None,The variation in ChatGPT usage across occupati...,1,1,1,8.252245,ea8552d8-401a-4880-a037-7004adc8398f,ffecf6b1-5357-4857-8647-29032af32305
8,How does the fact that ChatGPT has reached ove...,"Yo, peep this—ChatGPT hitting a colossal 700 m...",None,"By July 2025, ChatGPT had been used weekly by ...",1,1,1,4.385899,19ddd676-6939-476b-8516-bd1dfcdf948e,4e927d01-4835-4aa9-b2fb-fe41bb3fd8d7
9,What is Appendix D about in relation to ChatGP...,"Yo, Appendix D is the ultimate deep dive treas...",None,Appendix D presents variation in ChatGPT usage...,1,0,1,2.980493,ecbee565-4bd7-4f1f-9c07-3134263e7a11,a9b0882b-5eab-4994-ab70-4241c361d794


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

#### ANSWER

Dopeness improved dramatically because the prompt explicitly told the LLM to be engaging and avoid generic responses—the model simply followed those instructions.
Helpfulness barely changed because even with fewer chunks retrieved (4 vs 10), the system still had enough context to answer most questions adequately.
The main driver was prompt engineering—telling the model what tone to use directly shaped the output quality.

![](Langchain_comparison.png)
Ref: (https://smith.langchain.com/o/bb67da1a-f981-488f-9c09-ce7a028c911d/datasets/cd6ca13d-afd1-4e2a-a777-4062688fa96d/compare?selectedSessions=5537da3f-7c64-411e-a641-538327a63654%2C366d3a91-6703-458b-89c9-63efb56e6745&baseline=5537da3f-7c64-411e-a641-538327a63654&textDisplayMode=compact&compare-experiment-tab=0)

In [45]:
import pandas as pd
df_comp = pd.read_csv('crushing-part-32_warm-collar-60.csv', usecols=['inputs','crushing-part-32_helpfulness', 'warm-collar-60_helpfulness',
       'crushing-part-32_correctness', 'warm-collar-60_correctness', 'crushing-part-32_dopeness', 'warm-collar-60_dopeness'])

display(df_comp)



,inputs,crushing-part-32_helpfulness,warm-collar-60_helpfulness,crushing-part-32_correctness,warm-collar-60_correctness,crushing-part-32_dopeness,warm-collar-60_dopeness
0,"{""question"": ""How does the fact that ChatGPT h...",Y,Y,CORRECT,CORRECT,N,Y
1,"{""question"": ""Whi is the date in July 2025 tha...",Y,Y,CORRECT,CORRECT,N,Y
2,"{""question"": ""whats hapening in July 2025 with...",Y,Y,CORRECT,CORRECT,N,Y
3,"{""question"": ""How do work and non-work usag di...",Y,Y,CORRECT,CORRECT,N,Y
4,"{""question"": ""How does the rapid growth of Cha...",Y,Y,CORRECT,CORRECT,N,Y
5,"{""question"": ""What does Bick et al., 2024 say ...",N,Y,INCORRECT,CORRECT,N,Y
6,"{""question"": ""how much ChatGPT messages are wo...",Y,N,CORRECT,CORRECT,N,Y
7,"{""question"": ""Considering the rapid growth of ...",Y,Y,CORRECT,CORRECT,N,Y
8,"{""question"": ""How does the impact of occupatio...",Y,Y,CORRECT,CORRECT,N,Y
9,"{""question"": ""How do the variation in ChatGPT ...",Y,Y,CORRECT,CORRECT,N,Y
